<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent; border-radius: 14px; padding: 18px 22px; margin: 12px 0;
  font-size: 36px; font-weight: 600; color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25); background-clip: padding-box; position: relative;
">
  <div style="position: absolute; inset: 0; padding: 4px; border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor; mask-composite: exclude; pointer-events: none;"></div>
  <b>00. Structural Time Series Models with TFP</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [Introduction to Structural Time Series](#-part-i-introduction)**
    - 1.1 [What Are STS Models?](#what-are-sts-models)
    - 1.2 [Components of a Time Series](#components)
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [Building STS Models with tfp.sts](#-part-iii-building-sts)**
    - 3.1 [Local Linear Trend](#local-linear-trend)
    - 3.2 [Seasonal Component](#seasonal-component)
    - 3.3 [Combining Components](#combining-components)
- **4. [Fitting and Forecasting](#-part-iv-fitting-and-forecasting)**
    - 4.1 [Parameter Estimation via VI](#parameter-estimation)
    - 4.2 [Probabilistic Forecasting](#probabilistic-forecasting)
- **5. [Real-World Example: CO2 Concentrations](#-part-v-real-world-example)**
- **6. [Key Takeaways](#-part-vi-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. Introduction to Structural Time Series</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.1. What Are STS Models?</span>

**Structural Time Series (STS)** models decompose a time series into interpretable components:

$$y_t = \text{Trend}_t + \text{Seasonal}_t + \text{Regression}_t + \epsilon_t$$

| **Component** | **Description** | **TFP Class** |
|--------------|----------------|---------------|
| Local Linear Trend | Evolving level + slope | `tfp.sts.LocalLinearTrend` |
| Seasonal | Repeating periodic patterns | `tfp.sts.Seasonal` |
| Autoregressive | Short-term correlations | `tfp.sts.Autoregressive` |
| Linear Regression | Exogenous covariates | `tfp.sts.LinearRegression` |
| Semi-Local Linear Trend | Mean-reverting trend | `tfp.sts.SemiLocalLinearTrend` |

**Key advantage over traditional methods**: STS models provide **probabilistic forecasts** with calibrated uncertainty bands.

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

tfd = tfp.distributions
sts = tfp.sts

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Building STS Models with tfp.sts</span>

In [ ]:
# Generate synthetic time series with trend + seasonality
np.random.seed(42)
T = 200  # Total time steps
t = np.arange(T).astype(np.float32)

# Components
trend = 0.05 * t + 2.0
seasonal = 3.0 * np.sin(2 * np.pi * t / 12)  # Monthly seasonality
noise = np.random.normal(0, 0.5, T)

y = (trend + seasonal + noise).astype(np.float32)

# Split into train/test
train_size = 170
y_train = y[:train_size]
y_test = y[train_size:]

print(f"Training: {train_size} steps, Testing: {T - train_size} steps")

# Visualize
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(t[:train_size], y_train, color='#3b82f6', linewidth=1.5, label='Training')
ax.plot(t[train_size:], y_test, color='#ef4444', linewidth=1.5, label='Test (to predict)')
ax.axvline(x=train_size, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Time', fontsize=13)
ax.set_ylabel('Value', fontsize=13)
ax.set_title('Synthetic Time Series: Trend + Seasonality + Noise', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. Building the Model</span>

In [ ]:
# Build the STS model
observed_time_series = tf.constant(y_train[..., tf.newaxis])

# Define components
trend_component = sts.LocalLinearTrend(
    observed_time_series=observed_time_series,
    name='trend'
)

seasonal_component = sts.Seasonal(
    num_seasons=12,
    observed_time_series=observed_time_series,
    name='seasonal'
)

# Combine into a full model
model = sts.Sum(
    components=[trend_component, seasonal_component],
    observed_time_series=observed_time_series
)

print("STS Model Components:")
for component in model.components:
    print(f"  - {component.name}")
print(f"\nTotal parameters: {sum(p.shape.num_elements() for p in model.parameters)}")
for param in model.parameters:
    print(f"  {param.name}: shape={param.prior.batch_shape}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. Fitting and Forecasting</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.1. Parameter Estimation via Variational Inference</span>

In [ ]:
# Fit the model using Variational Inference
variational_posteriors = tfp.sts.build_factored_surrogate_posterior(
    model=model
)

# Optimize
num_variational_steps = 200

@tf.function(experimental_compile=True)
def train():
    return tfp.vi.fit_surrogate_posterior(
        target_log_prob_fn=model.joint_distribution(
            observed_time_series=observed_time_series
        ).log_prob,
        surrogate_posterior=variational_posteriors,
        optimizer=tf.optimizers.Adam(learning_rate=0.1),
        num_steps=num_variational_steps
    )

elbo_losses = train()

plt.figure(figsize=(10, 4))
plt.plot(elbo_losses.numpy(), color='#3b82f6', linewidth=2)
plt.xlabel('VI Step', fontsize=13)
plt.ylabel('Negative ELBO', fontsize=13)
plt.title('Variational Inference: ELBO Optimization', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.2. Probabilistic Forecasting</span>

In [ ]:
# Draw posterior samples
num_forecast_steps = T - train_size
posterior_samples = variational_posteriors.sample(50)

# Generate forecast distribution
forecast_dist = tfp.sts.forecast(
    model=model,
    observed_time_series=observed_time_series,
    parameter_samples=posterior_samples,
    num_steps_forecast=num_forecast_steps
)

forecast_mean = forecast_dist.mean().numpy()[..., 0]
forecast_std = forecast_dist.stddev().numpy()[..., 0]

# Average over posterior samples
mean_forecast = forecast_mean.mean(axis=0)
std_forecast = forecast_std.mean(axis=0)

# Visualize
fig, ax = plt.subplots(figsize=(16, 7))

# Training data
ax.plot(t[:train_size], y_train, color='#3b82f6', linewidth=1.5, label='Training data')

# Test data
ax.plot(t[train_size:], y_test, color='#1e293b', linewidth=2, label='True (held-out)')

# Forecast
forecast_t = t[train_size:]
ax.plot(forecast_t, mean_forecast, color='#ef4444', linewidth=2.5, label='Forecast mean')
ax.fill_between(forecast_t, mean_forecast - 2*std_forecast, mean_forecast + 2*std_forecast,
                alpha=0.2, color='#ef4444', label='±2σ forecast interval')
ax.fill_between(forecast_t, mean_forecast - std_forecast, mean_forecast + std_forecast,
                alpha=0.3, color='#ef4444')

ax.axvline(x=train_size, color='gray', linestyle='--', alpha=0.7, label='Forecast start')
ax.set_xlabel('Time', fontsize=14)
ax.set_ylabel('Value', fontsize=14)
ax.set_title('Probabilistic Forecast with Structural Time Series Model',
             fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Component decomposition
component_dists = tfp.sts.decompose_by_component(
    model=model,
    observed_time_series=observed_time_series,
    parameter_samples=posterior_samples
)

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Original
axes[0].plot(t[:train_size], y_train, color='#3b82f6', linewidth=1.5)
axes[0].set_title('Observed Time Series', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Value', fontsize=12)

# Trend
trend_mean = component_dists['trend'].mean().numpy().mean(axis=0)[:, 0]
trend_std = component_dists['trend'].stddev().numpy().mean(axis=0)[:, 0]
axes[1].plot(t[:train_size], trend_mean, color='#ef4444', linewidth=2)
axes[1].fill_between(t[:train_size], trend_mean - 2*trend_std, trend_mean + 2*trend_std,
                     alpha=0.2, color='#ef4444')
axes[1].set_title('Decomposed: Trend Component', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Trend', fontsize=12)

# Seasonal
seasonal_mean = component_dists['seasonal'].mean().numpy().mean(axis=0)[:, 0]
seasonal_std = component_dists['seasonal'].stddev().numpy().mean(axis=0)[:, 0]
axes[2].plot(t[:train_size], seasonal_mean, color='#22c55e', linewidth=2)
axes[2].fill_between(t[:train_size], seasonal_mean - 2*seasonal_std, seasonal_mean + 2*seasonal_std,
                     alpha=0.2, color='#22c55e')
axes[2].set_title('Decomposed: Seasonal Component', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Seasonal', fontsize=12)
axes[2].set_xlabel('Time', fontsize=13)

plt.suptitle('Bayesian Component Decomposition', fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| `tfp.sts` | High-level API for structural time series |
| Components | Trend, Seasonal, Autoregressive, Regression |
| Inference | Variational inference or HMC for parameter estimation |
| Forecasting | `tfp.sts.forecast` gives full predictive distribution |
| Decomposition | `decompose_by_component` separates trend/seasonal/etc |
| Uncertainty | Forecast intervals widen with horizon — natural & calibrated |

### Next Steps
- **Notebook 01**: Bayesian Time Series Forecasting with real datasets
- **Notebook 02**: Deep Probabilistic Time Series (RNNs + TFP)